# MCDI502 · Semana 3 · Evaluación Formativa 3
## Integración de OpenMeteo y AQICN con Apache Spark

**Magíster en Ciencia de Datos e Inteligencia Artificial — Grupo 2**  
**Entorno:** Visual Studio Code · Windows · Python 3.13 · Java 17 · PySpark 4.0.5

Cuaderno adaptado del archivo oficial `MCDI502_S3_EvaluacionFormativa(1).ipynb`. Contiene las siete partes de la guía: configuración, dos ingestas (5 registros por fuente), dos columnas calculadas, tres consultas SQL e integración con `UNION ALL`.

**Importante:** se consultan APIs reales; el cuaderno no inventa datos. AQICN requiere un token válido y puede devolver contaminantes faltantes. Las cinco consultas a AQICN son capturas sucesivas de una estación, no cinco mediciones históricas independientes.

## 1. Configuración de VS Code y Spark

Antes de abrir el notebook, en PowerShell crea la unidad virtual para evitar problemas con los espacios y acentos en la ruta del Magíster:

```powershell
$proyecto = 'C:\Users\eliza\Desktop\ESTUDIOS\MAGÍSTER\3 GESTIÓN DE DATOS Y TECNOLOGÍAS BIG DATA\Formativa 3 - APIs y Spark'
subst S: "$proyecto"
```

Selecciona el kernel `.venv` de esta carpeta. **No ejecutes comandos de instalación de Google Colab.** Si reinicias Windows, vuelve a crear `S:`. Las dependencias ya instaladas son `pyspark==4.0.5`, `pandas`, `requests` e `ipykernel`.

In [1]:
import os
import sys
from pathlib import Path
import requests
from datetime import datetime, timezone
from pyspark.sql import SparkSession, functions as F, types as T

# Spark requiere esta ruta corta en Windows para localizar correctamente los JAR.
SPARK_HOME_CORTO = r"S:\.venv\Lib\site-packages\pyspark"
PYTHON_CORTO = r"S:\.venv\Scripts\python.exe"
if not Path(SPARK_HOME_CORTO).is_dir():
    raise RuntimeError("No existe la unidad S:. Ejecuta 'subst S: <ruta del proyecto>' en PowerShell antes de iniciar el notebook.")
os.environ["SPARK_HOME"] = SPARK_HOME_CORTO
os.environ["PYSPARK_PYTHON"] = PYTHON_CORTO
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = (SparkSession.builder.master("local[2]")
         .appName("MCDI502_Formativa3_Grupo2")
         .config("spark.sql.shuffle.partitions", "2")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Python:", sys.version.split()[0])
print("Spark:", spark.version)
print("Prueba:", spark.range(5).count(), "registros")

Python: 3.13.17
Spark: 4.0.5
Prueba: 5 registros


## 2. Ingesta de OpenMeteo (5 registros)

Se usa la ubicación del ejemplo docente: **Viña del Mar, Chile** (latitud -33.02457; longitud -71.55183). Para obtener cinco observaciones con horas distintas, se consulta el pronóstico **horario** y se seleccionan las primeras cinco horas disponibles desde el momento de la consulta. Esto evita tratar cinco solicitudes idénticas de condiciones actuales como si fueran observaciones diferentes.

Variables: `temperature_2m`, `relative_humidity_2m`, `precipitation`, `wind_speed_10m` y `uv_index`.

In [2]:
LATITUD, LONGITUD = -33.02457, -71.55183
VARIABLES_METEO = ["temperature_2m", "relative_humidity_2m", "precipitation", "wind_speed_10m", "uv_index"]
url_meteo = "https://api.open-meteo.com/v1/forecast"
respuesta = requests.get(url_meteo, params={
    "latitude": LATITUD, "longitude": LONGITUD,
    "hourly": ",".join(VARIABLES_METEO),
    "forecast_days": 2, "timezone": "America/Santiago"
}, timeout=30)
respuesta.raise_for_status()
contenido = respuesta.json()
horas = contenido.get("hourly", {}).get("time", [])
if not horas:
    raise RuntimeError("OpenMeteo no devolvió registros horarios.")

# Elegir las primeras cinco horas de la respuesta, con sus fechas reales.
indices = list(range(min(5, len(horas))))
registros_meteo = []
for i in indices:
    fila = {"tiempo": horas[i]}
    for variable in VARIABLES_METEO:
        valores = contenido["hourly"].get(variable, [])
        valor = valores[i] if i < len(valores) else None
        fila[variable] = float(valor) if valor is not None else None
    registros_meteo.append(fila)

esquema_meteo = T.StructType([
    T.StructField("tiempo", T.StringType(), False),
    *[T.StructField(nombre, T.DoubleType(), True) for nombre in VARIABLES_METEO]
])
df_openmeteo = spark.createDataFrame(registros_meteo, esquema_meteo)
assert df_openmeteo.count() == 5, "La actividad exige cinco registros OpenMeteo"
print("OpenMeteo:", df_openmeteo.count(), "registros horarios")
df_openmeteo.show(5, truncate=False)
df_openmeteo.printSchema()

OpenMeteo: 5 registros horarios
+----------------+--------------+--------------------+-------------+--------------+--------+
|tiempo          |temperature_2m|relative_humidity_2m|precipitation|wind_speed_10m|uv_index|
+----------------+--------------+--------------------+-------------+--------------+--------+
|2026-11-14T00:00|16.9          |92.0                |0.0          |3.7           |0.0     |
|2026-11-14T01:00|16.3          |95.0                |0.0          |3.5           |0.0     |
|2026-11-14T02:00|16.2          |97.0                |0.0          |3.1           |0.0     |
|2026-11-14T03:00|16.8          |98.0                |0.0          |5.3           |0.0     |
|2026-11-14T04:00|16.6          |99.0                |0.0          |7.2           |0.0     |
+----------------+--------------+--------------------+-------------+--------------+--------+

root
 |-- tiempo: string (nullable = false)
 |-- temperature_2m: double (nullable = true)
 |-- relative_humidity_2m: double (nulla

## 3. Ingesta de AQICN (5 capturas)

La guía utiliza la API de AQICN con la estación `santiago` y un token de ejemplo `demo`. Ese token puede estar limitado o no autorizar esa estación. **Obtén un token personal desde [AQICN](https://aqicn.org/data-platform/token/) y configúralo como variable de entorno `AQICN_TOKEN`**, o escríbelo temporalmente en la variable de la celda sin guardarlo en GitHub.

El endpoint devuelve valores de índice de calidad del aire (`iaqi`), que **no necesariamente son concentraciones físicas**. El promedio pedido por la guía es un promedio aritmético de esos cuatro valores y solo se calcula cuando están los cuatro. No debe interpretarse como un índice oficial de contaminación.

Las cinco capturas se realizan en momentos sucesivos y pueden tener el mismo valor porque AQICN no necesariamente actualiza cada segundo.

In [3]:
import time

# Opción recomendada: configurar AQICN_TOKEN en el entorno antes de abrir VS Code.
TOKEN_AQICN = os.getenv("AQICN_TOKEN", "demo").strip()
ESTACION_AQICN = "santiago"
url_aqicn = f"https://api.waqi.info/feed/{ESTACION_AQICN}/"
registros_aqicn = []

for intento in range(5):
    respuesta = requests.get(url_aqicn, params={"token": TOKEN_AQICN}, timeout=30)
    respuesta.raise_for_status()
    datos = respuesta.json()
    if datos.get("status") != "ok":
        raise RuntimeError(
            "AQICN rechazó la consulta: " + str(datos.get("data")) +
            ". Configura un token válido en AQICN_TOKEN y vuelve a ejecutar esta celda."
        )
    iaqi = datos.get("data", {}).get("iaqi", {})
    def extraer(clave):
        v = iaqi.get(clave, {}).get("v")
        try:
            return float(v) if v is not None else None
        except (ValueError, TypeError):
            return None
    registros_aqicn.append({
        "tiempo_consulta": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "PM2_5": extraer("pm25"), "PM10": extraer("pm10"),
        "O3": extraer("o3"), "NO2": extraer("no2")
    })
    if intento < 4:
        time.sleep(1)

esquema_aqicn = T.StructType([
    T.StructField("tiempo_consulta", T.StringType(), False),
    *[T.StructField(x, T.DoubleType(), True) for x in ["PM2_5", "PM10", "O3", "NO2"]]
])
df_aqicn = spark.createDataFrame(registros_aqicn, esquema_aqicn)
assert df_aqicn.count() == 5, "La actividad exige cinco registros AQICN"
print("AQICN:", df_aqicn.count(), "capturas de la estación", ESTACION_AQICN)
df_aqicn.show(5, truncate=False)
df_aqicn.printSchema()
print("Advertencia: los valores pueden repetirse porque son capturas de la misma estación.")

AQICN: 5 capturas de la estación santiago
+-------------------------+-----+----+----+---+
|tiempo_consulta          |PM2_5|PM10|O3  |NO2|
+-------------------------+-----+----+----+---+
|2026-11-14T22:04:37+00:00|117.0|44.0|17.5|9.2|
|2026-11-14T22:04:39+00:00|117.0|44.0|17.5|9.2|
|2026-11-14T22:04:42+00:00|117.0|44.0|17.5|9.2|
|2026-11-14T22:04:44+00:00|117.0|44.0|17.5|9.2|
|2026-11-14T22:04:46+00:00|117.0|44.0|17.5|9.2|
+-------------------------+-----+----+----+---+

root
 |-- tiempo_consulta: string (nullable = false)
 |-- PM2_5: double (nullable = true)
 |-- PM10: double (nullable = true)
 |-- O3: double (nullable = true)
 |-- NO2: double (nullable = true)

Advertencia: los valores pueden repetirse porque son capturas de la misma estación.


## 4. Transformaciones: dos columnas calculadas

1. `temperatura_fahrenheit = (temperature_2m × 9/5) + 32`, con dos decimales.
2. `promedio_contaminantes = (PM2_5 + PM10 + O3 + NO2) / 4`, con dos decimales. Si falta alguno de los cuatro valores, el promedio queda nulo para evitar presentar un promedio de cuatro componentes incompleto.

In [4]:
df_openmeteo = df_openmeteo.withColumn(
    "temperatura_fahrenheit", F.round(F.col("temperature_2m") * 9 / 5 + 32, 2)
)
df_aqicn = df_aqicn.withColumn(
    "promedio_contaminantes",
    F.round((F.col("PM2_5") + F.col("PM10") + F.col("O3") + F.col("NO2")) / 4, 2)
)
print("Temperatura en Fahrenheit")
df_openmeteo.select("tiempo", "temperature_2m", "temperatura_fahrenheit").show(truncate=False)
print("Promedio de contaminantes")
df_aqicn.select("PM2_5", "PM10", "O3", "NO2", "promedio_contaminantes").show(truncate=False)
if df_aqicn.filter(F.col("promedio_contaminantes").isNull()).count() > 0:
    print("AVISO: AQICN no entregó todos los contaminantes en una o más capturas; el promedio queda NULL.")

Temperatura en Fahrenheit
+----------------+--------------+----------------------+
|tiempo          |temperature_2m|temperatura_fahrenheit|
+----------------+--------------+----------------------+
|2026-11-14T00:00|16.9          |62.42                 |
|2026-11-14T01:00|16.3          |61.34                 |
|2026-11-14T02:00|16.2          |61.16                 |
|2026-11-14T03:00|16.8          |62.24                 |
|2026-11-14T04:00|16.6          |61.88                 |
+----------------+--------------+----------------------+

Promedio de contaminantes
+-----+----+----+---+----------------------+
|PM2_5|PM10|O3  |NO2|promedio_contaminantes|
+-----+----+----+---+----------------------+
|117.0|44.0|17.5|9.2|46.93                 |
|117.0|44.0|17.5|9.2|46.93                 |
|117.0|44.0|17.5|9.2|46.93                 |
|117.0|44.0|17.5|9.2|46.93                 |
|117.0|44.0|17.5|9.2|46.93                 |
+-----+----+----+---+----------------------+



## 5. Spark SQL: tres consultas solicitadas

Se registran vistas temporales y se ejecutan las consultas de temperatura máxima, mínima y media; promedio general de contaminantes; y humedad relativa promedio.

In [5]:
df_openmeteo.createOrReplaceTempView("openmeteo_data")
df_aqicn.createOrReplaceTempView("aqicn_data")

result1 = spark.sql("""
SELECT MAX(temperature_2m) AS temp_maxima,
       MIN(temperature_2m) AS temp_minima,
       ROUND(AVG(temperature_2m), 2) AS temp_promedio
FROM openmeteo_data
""")
print("Consulta 1: Temperatura máxima, mínima y promedio")
result1.show(truncate=False)

Consulta 1: Temperatura máxima, mínima y promedio
+-----------+-----------+-------------+
|temp_maxima|temp_minima|temp_promedio|
+-----------+-----------+-------------+
|16.9       |16.2       |16.56        |
+-----------+-----------+-------------+



In [6]:
result2 = spark.sql("""
SELECT ROUND(AVG(promedio_contaminantes), 2) AS promedio_general
FROM aqicn_data
""")
print("Consulta 2: Promedio general de contaminantes")
result2.show(truncate=False)
if result2.first()["promedio_general"] is None:
    print("Resultado NULL: no hay capturas con los cuatro contaminantes disponibles.")

Consulta 2: Promedio general de contaminantes
+----------------+
|promedio_general|
+----------------+
|46.93           |
+----------------+



In [7]:
result3 = spark.sql("""
SELECT ROUND(AVG(relative_humidity_2m), 2) AS humedad_promedio
FROM openmeteo_data
""")
print("Consulta 3: Humedad relativa promedio")
result3.show(truncate=False)

Consulta 3: Humedad relativa promedio
+----------------+
|humedad_promedio|
+----------------+
|96.2            |
+----------------+



## 6. Integración básica con `UNION ALL`

Siguiendo la guía docente, se crea un DataFrame unificado **de resumen por fuente**. Esta integración concatena dos conteos; no mezcla directamente las columnas meteorológicas y de contaminación.

In [8]:
df_unificado = spark.sql("""
SELECT 'OpenMeteo' AS fuente, COUNT(*) AS registros FROM openmeteo_data
UNION ALL
SELECT 'AQICN' AS fuente, COUNT(*) AS registros FROM aqicn_data
""")
print("Resumen de registros por fuente")
df_unificado.show(truncate=False)
assert df_unificado.count() == 2
assert df_unificado.agg(F.sum("registros")).first()[0] == 10

Resumen de registros por fuente
+---------+---------+
|fuente   |registros|
+---------+---------+
|OpenMeteo|5        |
|AQICN    |5        |
+---------+---------+



## 7. Validaciones y resumen final

Las validaciones comprueban las cantidades requeridas y la presencia de columnas. **El éxito del notebook depende de respuestas reales de ambas APIs y de un token AQICN autorizado**; no se incluyen salidas inventadas ni capturas simuladas.

In [9]:
assert df_openmeteo.count() == 5
assert df_aqicn.count() == 5
assert "temperatura_fahrenheit" in df_openmeteo.columns
assert "promedio_contaminantes" in df_aqicn.columns

print("=" * 70)
print("MCDI502 · EVALUACIÓN FORMATIVA · SEMANA 3")
print("=" * 70)
print("OpenMeteo: 5 registros horarios")
print("AQICN: 5 capturas de una estación")
print("Columnas calculadas: 2")
print("Consultas Spark SQL: 3")
print("Integración UNION ALL: 2 filas de resumen; 10 registros de origen")
print("Validaciones de estructura y conteo: correctas")
print("Nota: los resultados dependen de la disponibilidad y cobertura de las APIs.")

MCDI502 · EVALUACIÓN FORMATIVA · SEMANA 3
OpenMeteo: 5 registros horarios
AQICN: 5 capturas de una estación
Columnas calculadas: 2
Consultas Spark SQL: 3
Integración UNION ALL: 2 filas de resumen; 10 registros de origen
Validaciones de estructura y conteo: correctas
Nota: los resultados dependen de la disponibilidad y cobertura de las APIs.


### Reflexión breve

En esta actividad conecté dos fuentes externas con formatos diferentes y transformé sus respuestas en DataFrames de Spark. Apliqué cálculos derivados y consultas SQL para resumir los datos, y utilicé `UNION ALL` para consolidar el número de registros por fuente. Identifiqué dos limitaciones importantes: las capturas sucesivas de AQICN pueden repetir una misma medición, y la ausencia de algún contaminante impide calcular un promedio completo de cuatro variables. Esto muestra por qué la calidad y la interpretación de los datos son tan importantes como su procesamiento.

**Próxima etapa:** Taller II / Sumativa 3, que incorpora MongoDB y MySQL mediante API REST, además de las dos fuentes trabajadas aquí.

## 8. Cierre opcional de la sesión

Ejecuta la siguiente celda solo cuando termines de revisar los resultados. Si vuelves a trabajar en el notebook, reinicia el kernel y ejecuta las celdas en orden.

In [10]:
spark.stop()
print("Sesión Spark finalizada.")

Sesión Spark finalizada.
